In [1]:
%pip install langgraph
%pip install langsmith
%pip install IPython
%pip install langchain_tavily

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import Annotated 
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END 
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


import os
import dotenv
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

model = init_chat_model(
    "gemini-3.5-flash",
    model_provider="google_genai"

)


In [3]:
class State(TypedDict):
    """State of the chatbot conversation. Messages are not replaced but added in the list of the messages."""
    messages: Annotated[list[str], add_messages] #in LangGraph is used to attach reducer functions to state fields. The reducer tells LangGraph how to merge updates from multiple nodes or graph branches.



In [4]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)

tool.invoke("What is the capital of France?")

{'query': 'What is the capital of France?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.facebook.com/groups/302915185514582/posts/740821111723985',
   'title': 'What is the capital of France? - Facebook',
   'content': "PARIS it is the capital and largest city of France. Located on the Seine in the country's north, it is a major cultural and political centre of",
   'score': 0.910998,
   'raw_content': None},
  {'url': 'https://home.adelphi.edu/~ca19535/page%204.html',
   'title': 'Paris facts: the capital of France in history',
   'content': 'Paris is the **capital of [France](http://www.parisdigest.com/famous_places_in_france.htm)**, the largest country of [Europe](http://www.parisdigest.com/famous_places_in_europe) with 550 000 km2 (65 millions inhabitants). Founded more than 2000 years ago, Paris is a modern and vibrant city with significant [commercial](http://www.parisdigest.com/menus/shopping.htm), [cultural](http://www.parisdi

In [5]:

def multiply(a: int , b: int ) -> int:
    """Multiplies two numbers.
    
    Args:
        a: The first number.
        b: The second number.
    Returns:
        The product of a and b."""
    
    return a*b

In [6]:
tools = [tool, multiply]

In [7]:
llm_with_tools = model.bind_tools(tools)

Key 'additionalProperties' is not supported in schema, ignoring


In [8]:
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    {"tools": "tools", "end": END}
)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()
